# Association Rule Mining Assignment
## Part A – Reasoning (30 pts)

### Why FP Analysis for All Items in Walmart is Not Possible

**Scenario / Assumptions:**
- Walmart stocks approximately **100,000 unique products (items)**.
- A typical transaction (basket) contains on average **20–30 items**.
- Walmart processes roughly **1 million transactions per day** (globally).

---

#### Combinatorial Explosion

The total number of possible itemsets (non-empty subsets) of $n$ items is:

$$2^n - 1$$

With $n = 100{,}000$ Walmart items:

$$2^{100{,}000} - 1 \approx 10^{30{,}103}$$

For comparison, the estimated number of atoms in the observable universe is only $\approx 10^{80}$.  
This number of candidate itemsets is **astronomically larger** than anything physically enumerable.

Even if we restrict to **just 2-itemsets** (pairs), the number of combinations is:

$$\binom{100{,}000}{2} = \frac{100{,}000 \times 99{,}999}{2} \approx 5 \times 10^9 \text{ (5 billion pairs)}$$

And for 3-itemsets:

$$\binom{100{,}000}{3} = \frac{100{,}000 \times 99{,}999 \times 99{,}998}{6} \approx 1.67 \times 10^{14} \text{ (167 trillion triplets)}$$

The number grows factorially with itemset size, making full enumeration impossible.

---

#### Space Complexity

**FP-tree memory requirements:**
- With 1 million transactions of ~25 items each, the FP-tree could have up to **25 million nodes**.
- Each node stores: item ID (4 bytes), count (4 bytes), parent pointer (8 bytes), child pointers (~8 bytes) = ~30 bytes/node.
- Estimated FP-tree size: $25 \times 10^6 \times 30 \approx 750$ MB just for the top-level tree.
- Generating **conditional FP-trees** for 100,000 items is infeasible:
  - Worst-case space: $O(n \times |DB|)$ => $100{,}000 \times 25 \times 10^6 \times 30 \approx 75$ **TB**

---

#### Time Complexity

**Apriori-style candidate generation:**
- Time per level: $O(|DB| \times |C_k|)$
- At $k=2$: $10^6 \times 5 \times 10^9 = 5 \times 10^{15}$ operations => at $10^9$ ops/sec this is **~58 days** for level 2 alone.

**FP-Growth:**
- Must recursively mine conditional trees for each frequent item.
- Worst-case: $O(2^n)$ time—computationally intractable for $n = 100{,}000$.

---

#### Conclusion

Running FP analysis over all Walmart items is **mathematically and computationally infeasible** because:
1. The search space ($2^{100,000}$) dwarfs the number of atoms in the universe.
2. Space requirements would require petabytes of RAM.
3. Time requirements would exceed the age of the universe even on modern hardware.

In practice, retailers limit analysis to **top-selling items** (e.g., top 1,000–10,000), apply high minimum support thresholds, or use approximate/sampling methods.

---
## Part B – Applications (30 pts)
### Question 1: Evaluate Association Rules

#### Contingency Table

|           | ski  | not-ski | Row Total |
|-----------|------|---------|----------|
| **bike**  | 1000 | 1500    | 2500     |
| **not-bike** | 1200 | 300  | 1500     |
| **Col Total** | 2200 | 1800 | 4000   |

---

#### (a) Lift Measure between Bike and Ski

$$\text{Lift}(\text{bike} \Rightarrow \text{ski}) = \frac{P(\text{bike} \cap \text{ski})}{P(\text{bike}) \times P(\text{ski})}$$

$$P(\text{bike} \cap \text{ski}) = \frac{1000}{4000} = 0.25$$

$$P(\text{bike}) = \frac{2500}{4000} = 0.625 \qquad P(\text{ski}) = \frac{2200}{4000} = 0.55$$

$$\text{Lift} = \frac{0.25}{0.625 \times 0.55} = \frac{0.25}{0.34375} \approx \boxed{0.727}$$

**Interpretation:** Since Lift < 1, bike and ski are **negatively correlated**. Students who bike are *less* likely to ski than expected by chance.

---

#### (b) Is the Rule bike => not-ski Strong?

**Support** of (bike AND not-ski):
$$\text{support} = \frac{1500}{4000} = 0.375 = 37.5\%$$

Min support threshold = 20% => **37.5% >= 20% (PASS)**

**Confidence** of bike => not-ski:
$$\text{confidence} = \frac{P(\text{bike} \cap \overline{\text{ski}})}{P(\text{bike})} = \frac{1500/4000}{2500/4000} = \frac{1500}{2500} = 0.60 = 60\%$$

Min confidence threshold = 50% => **60% >= 50% (PASS)**

**Conclusion:** The rule bike => not-ski **IS a strong association rule** — it meets both thresholds.

---
### Question 2: Apriori by Hand

**Transactions:**

| TID | Items Bought        |
|-----|---------------------|
| T1  | {M, N, P, X, Z}    |
| T2  | {M, A, P, X, O}    |
| T3  | {D, N, P, X, Z}    |
| T4  | {M, O, C, P, Z}    |
| T5  | {M, K, I, Z, O}    |

**min_support = 50%** => minimum count = 0.5 x 5 = 2.5, so an itemset must appear in **at least 3 transactions**.

---

#### (a) Maximum Number of Possible Frequent Itemsets

Unique items: {M, N, P, X, Z, A, O, D, C, K, I} = **11 unique items**.

Maximum possible non-empty itemsets:
$$2^{11} - 1 = 2048 - 1 = \boxed{2047}$$

---

#### (b) Apriori Algorithm — Key Steps (min_support = 50%, min_count = 3)

**Step 1 — C1: Count all 1-itemsets:**

| Item | Count | Frequent? |
|------|-------|-----------|
| M    | 4     | YES       |
| N    | 2     | NO        |
| P    | 4     | YES       |
| X    | 3     | YES       |
| Z    | 4     | YES       |
| A    | 1     | NO        |
| O    | 3     | YES       |
| D    | 1     | NO        |
| C    | 1     | NO        |
| K    | 1     | NO        |
| I    | 1     | NO        |

**L1 (frequent 1-itemsets):** {M:4, P:4, X:3, Z:4, O:3}

---

**Step 2 — C2: Generate pairs from L1 and count:**

| Itemset  | Transactions     | Count | Frequent? |
|----------|------------------|-------|-----------|
| {M, P}   | T1, T2, T4       | 3     | YES       |
| {M, X}   | T1, T2           | 2     | NO        |
| {M, Z}   | T1, T4, T5       | 3     | YES       |
| {M, O}   | T2, T4, T5       | 3     | YES       |
| {P, X}   | T1, T2, T3       | 3     | YES       |
| {P, Z}   | T1, T3, T4       | 3     | YES       |
| {P, O}   | T2, T4           | 2     | NO        |
| {X, Z}   | T1, T3           | 2     | NO        |
| {X, O}   | T2               | 1     | NO        |
| {Z, O}   | T4, T5           | 2     | NO        |

**L2:** {M,P}, {M,Z}, {M,O}, {P,X}, {P,Z}

---

**Step 3 — C3: Generate 3-itemsets from L2 and prune:**

Join pairs sharing a common prefix:
- {M,P} + {M,Z} => {M,P,Z}: check subsets {P,Z} in L2? YES => valid candidate
- {M,P} + {M,O} => {M,P,O}: check {P,O} in L2? NO => **pruned**
- {M,Z} + {M,O} => {M,Z,O}: check {Z,O} in L2? NO => **pruned**
- {P,X} + {P,Z} => {P,X,Z}: check {X,Z} in L2? NO => **pruned**

Only candidate: {M, P, Z} => appears in T1 and T4 => count = **2** < 3 => NOT frequent

**L3 is empty. Algorithm terminates.**

---

**All Frequent Itemsets:**
- **1-itemsets:** {M}, {P}, {X}, {Z}, {O}
- **2-itemsets:** {M,P}, {M,Z}, {M,O}, {P,X}, {P,Z}

---

#### (c) Rounds of DB Scans and Total Candidates

- **Rounds of DB scans:** **3** (one per level: C1, C2, C3)
- **Total candidates:**
  - C1: 11 candidates
  - C2: 10 candidates
  - C3: 1 candidate
  - **Total = 22 candidates**

---
### Question 3: FP-Growth by Hand

Same dataset, **min_support = 50%** (min count = 3).

Frequent 1-itemsets ordered by decreasing frequency: **P:4, M:4, Z:4, X:3, O:3**

---

**Step 1 — Reorder each transaction (keep only frequent items, sorted by frequency):**

| TID | Original           | Filtered & Sorted |
|-----|--------------------|-------------------|
| T1  | {M, N, P, X, Z}   | P, M, Z, X        |
| T2  | {M, A, P, X, O}   | P, M, X, O        |
| T3  | {D, N, P, X, Z}   | P, Z, X           |
| T4  | {M, O, C, P, Z}   | P, M, Z, O        |
| T5  | {M, K, I, Z, O}   | M, Z, O           |

---

**Step 2 — Build the FP-Tree (insert transactions one by one):**

```
root
|-- P:4
|   |-- M:3
|   |   |-- Z:2
|   |   |   |-- X:1   (T1)
|   |   |   |-- O:1   (T4)
|   |   |-- X:1
|   |       |-- O:1   (T2)
|   |-- Z:1
|       |-- X:1       (T3)
|-- M:1               (T5, no P)
    |-- Z:1
        |-- O:1
```

**Header Table:**

| Item | Count |
|------|-------|
| P    | 4     |
| M    | 4     |
| Z    | 4     |
| X    | 3     |
| O    | 3     |

---

**Step 3 — Mine Conditional Pattern Bases (bottom-up):**

**Item O (count=3):**
- Conditional pattern base: {P,M,Z:1}, {P,M,X:1}, {M,Z:1}
- Conditional frequent items (count >= 3): M:3 only
- Frequent itemsets with O: **{O}, {M,O}**

**Item X (count=3):**
- Conditional pattern base: {P,M,Z:1}, {P,M:1}, {P,Z:1}
- Conditional frequent items: P:3
- Frequent itemsets with X: **{X}, {P,X}**

**Item Z (count=4):**
- Conditional pattern base: {P,M:2}, {P:1}, {M:1}
- Conditional frequent items: P:3, M:3; combined {P,M}: count=2 < 3
- Frequent itemsets with Z: **{Z}, {P,Z}, {M,Z}**

**Item M (count=4):**
- Conditional pattern base: {P:3}, {empty:1}
- Conditional frequent items: P:3
- Frequent itemsets with M: **{M}, {P,M}**

**Item P (count=4):**
- Conditional pattern base: {} (root only)
- Frequent itemsets with P: **{P}**

---

**All Frequent Itemsets from FP-Growth:**
- **1-itemsets:** {P}, {M}, {Z}, {X}, {O}
- **2-itemsets:** {M,O}, {P,X}, {P,Z}, {M,Z}, {P,M}

**Result matches Apriori exactly.** FP-Growth avoids candidate generation by compressing the database into an FP-tree and mining patterns recursively from conditional trees.

---
## Part C – Do Apriori with Python (40 pts)
### Setup: Install packages and load data

In [6]:
import pandas as pd
import numpy as np
import time
from apyori import apriori
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, apriori as mlxtend_apriori, association_rules

# Load the dataset: each row is a transaction, items comma-separated, no header
transactions = []
with open('store_data.csv', 'r') as f:
    for line in f:
        items = [item.strip() for item in line.strip().split(',') if item.strip()]
        if items:
            transactions.append(items)

print(f'Total transactions: {len(transactions)}')
print(f'Sample transaction 0: {transactions[0]}')
print(f'Sample transaction 1: {transactions[1]}')

Total transactions: 7501
Sample transaction 0: ['shrimp', 'almonds', 'avocado', 'vegetables mix', 'green grapes', 'whole weat flour', 'yams', 'cottage cheese', 'energy drink', 'tomato juice', 'low fat yogurt', 'green tea', 'honey', 'salad', 'mineral water', 'salmon', 'antioxydant juice', 'frozen smoothie', 'spinach', 'olive oil']
Sample transaction 1: ['burgers', 'meatballs', 'eggs']


In [7]:
# Helper: extract and display rules from apyori results
def display_rules(results, max_display=None):
    rules = []
    for result in results:
        support = result.support
        for stat in result.ordered_statistics:
            if len(stat.items_base) > 0:
                rules.append({
                    'Antecedent': ', '.join(list(stat.items_base)),
                    'Consequent': ', '.join(list(stat.items_add)),
                    'Support':    round(support, 4),
                    'Confidence': round(stat.confidence, 4),
                    'Lift':       round(stat.lift, 4)
                })
    df = pd.DataFrame(rules)
    print(f'Total rules found: {len(df)}')
    if max_display:
        return df.head(max_display)
    return df

### 2-Itemset Association Rules (~10 rules, max_length=2)

In [8]:
# 2-itemset rules: max_length=2, tuned to produce ~10 rules
results_2item = list(apriori(
    transactions,
    min_support=0.003,
    min_confidence=0.20,
    min_lift=3.0,
    max_length=2
))

rules_2item = display_rules(results_2item)
rules_2item

Total rules found: 9


,Antecedent,Consequent,Support,Confidence,Lift
0,light cream,chicken,0.0045,0.2906,4.8440
1,mushroom cream sauce,escalope,0.0057,0.3007,3.7908
2,pasta,escalope,0.0059,0.3729,4.7008
3,fromage blanc,honey,0.0033,0.2451,5.1643
4,herb & pepper,ground beef,0.0160,0.3235,3.2920
5,tomato sauce,ground beef,0.0053,0.3774,3.8407
6,light cream,olive oil,0.0032,0.2051,3.1147
7,whole wheat pasta,olive oil,0.0080,0.2715,4.1224
8,pasta,shrimp,0.0051,0.3220,4.5067


### 3+ Itemset Rules: 0 rules

In [9]:
# Very high thresholds => 0 rules
results_0 = list(apriori(
    transactions,
    min_support=0.10,
    min_confidence=0.90,
    min_lift=10.0,
    max_length=10
))
rules_0 = display_rules(results_0)
rules_0

Total rules found: 0


""


### 3+ Itemset Rules: ~10 rules

In [10]:
# Tuned to produce ~10 rules for 3+ itemsets
results_10 = list(apriori(
    transactions,
    min_support=0.002,
    min_confidence=0.20,
    min_lift=4.0,
    max_length=10
))
rules_10 = display_rules(results_10, max_display=10)
rules_10

Total rules found: 91


,Antecedent,Consequent,Support,Confidence,Lift
0,light cream,chicken,0.0045,0.2906,4.8440
1,pasta,escalope,0.0059,0.3729,4.7008
2,fromage blanc,honey,0.0033,0.2451,5.1643
3,whole wheat pasta,olive oil,0.0080,0.2715,4.1224
4,pasta,shrimp,0.0051,0.3220,4.5067
5,"herb & pepper, burgers",ground beef,0.0023,0.5484,5.5813
6,"frozen vegetables, cake",tomatoes,0.0031,0.2987,4.3676
7,"spaghetti, cereals",ground beef,0.0031,0.4600,4.6818
8,"ground beef, chicken",herb & pepper,0.0021,0.2254,4.5562
9,"herb & pepper, chicken",ground beef,0.0021,0.4324,4.4012


### 3+ Itemset Rules: ~30 rules

In [11]:
# Tuned to produce ~30 rules for 3+ itemsets
results_30 = list(apriori(
    transactions,
    min_support=0.002,
    min_confidence=0.20,
    min_lift=3.0,
    max_length=10
))
rules_30 = display_rules(results_30)
print('Showing first 5 rules:')
rules_30.head(5)

Total rules found: 347
Showing first 5 rules:


,Antecedent,Consequent,Support,Confidence,Lift
0,barbecue sauce,turkey,0.0025,0.2346,3.7516
1,extra dark chocolate,chicken,0.0028,0.2333,3.8894
2,light cream,chicken,0.0045,0.2906,4.8440
3,mushroom cream sauce,escalope,0.0057,0.3007,3.7908
4,pasta,escalope,0.0059,0.3729,4.7008


### 3+ Itemset Rules: ~100 rules

In [12]:
# Tuned to produce ~100 rules for 3+ itemsets
results_100 = list(apriori(
    transactions,
    min_support=0.002,
    min_confidence=0.15,
    min_lift=2.0,
    max_length=10
))
rules_100 = display_rules(results_100)
# No need to display rule details per instructions
print(f'Total 3+ itemset rules: {len(rules_100)}')

Total rules found: 1663
Total 3+ itemset rules: 1663


---
### 2. FP-Growth vs Apriori Performance Comparison

In [13]:
# Encode transactions into boolean DataFrame for mlxtend
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)
print(f'Encoded: {df_encoded.shape[0]} transactions x {df_encoded.shape[1]} unique items')

Encoded: 7501 transactions x 119 unique items


In [14]:
min_sup = 0.002  # same threshold for fair comparison

# --- Apriori (mlxtend) ---
t0 = time.time()
freq_apriori = mlxtend_apriori(df_encoded, min_support=min_sup, use_colnames=True, max_len=4)
rules_ap = association_rules(freq_apriori, metric='lift', min_threshold=2.0)
t_apriori = time.time() - t0

print(f'Apriori   | Frequent itemsets: {len(freq_apriori):>4} | Rules: {len(rules_ap):>4} | Time: {t_apriori:.3f}s')

Apriori   | Frequent itemsets: 2435 | Rules: 4596 | Time: 0.895s


In [15]:
# --- FP-Growth (mlxtend) ---
t0 = time.time()
freq_fp = fpgrowth(df_encoded, min_support=min_sup, use_colnames=True, max_len=4)
rules_fp = association_rules(freq_fp, metric='lift', min_threshold=2.0)
t_fp = time.time() - t0

print(f'FP-Growth | Frequent itemsets: {len(freq_fp):>4} | Rules: {len(rules_fp):>4} | Time: {t_fp:.3f}s')

FP-Growth | Frequent itemsets: 2435 | Rules: 4596 | Time: 0.159s


In [16]:
# Summary comparison table
summary = pd.DataFrame({
    'Algorithm':         ['Apriori', 'FP-Growth'],
    'Frequent Itemsets': [len(freq_apriori), len(freq_fp)],
    'Rules Found':       [len(rules_ap), len(rules_fp)],
    'Time (s)':          [round(t_apriori, 3), round(t_fp, 3)]
})
print(summary.to_string(index=False))

# Top 5 rules by lift from FP-Growth
top_rules = rules_fp.sort_values('lift', ascending=False).head(5).copy()
top_rules['antecedents'] = top_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
top_rules['consequents'] = top_rules['consequents'].apply(lambda x: ', '.join(list(x)))
print('\nTop 5 rules by Lift (FP-Growth):')
print(top_rules[['antecedents','consequents','support','confidence','lift']].to_string(index=False))

Algorithm  Frequent Itemsets  Rules Found  Time (s)
  Apriori               2435         4596     0.895
FP-Growth               2435         4596     0.159

Top 5 rules by Lift (FP-Growth):
                   antecedents                    consequents  support  confidence      lift
                         pasta escalope, mushroom cream sauce 0.002533    0.161017 28.088096
escalope, mushroom cream sauce                          pasta 0.002533    0.441860 28.088096
               escalope, pasta           mushroom cream sauce 0.002533    0.431818 22.650826
          mushroom cream sauce                escalope, pasta 0.002533    0.132867 22.650826
                      escalope    pasta, mushroom cream sauce 0.002533    0.031933 11.976387


### Apriori vs FP-Growth Discussion

| Aspect | Apriori | FP-Growth |
|---|---|---|
| **Candidate generation** | Explicit level-by-level generation and pruning | None -- patterns mined directly from FP-tree |
| **Database scans** | One scan per itemset length (multiple passes) | Exactly 2 scans (one to find frequent items, one to build tree) |
| **Memory** | Low if few candidates survive; high otherwise | Requires the FP-tree to fit in RAM |
| **Speed** | Slower due to repeated scans and large candidate sets | Generally faster -- no candidate generation overhead |
| **Results** | Identical frequent itemsets and rules | Identical (both are exact algorithms) |

**Observed:** FP-Growth is typically faster on real-world datasets because it compresses the entire database into a compact tree structure and extracts patterns without the generate-and-test overhead of Apriori. The speed advantage grows as the number of frequent items and itemset length increases.